# Optimize Threshold for GANBase Adaptive Sequencing

This notebook provides an automated pipeline to help users select the optimal classification threshold based on specific host and target reference data under predefined mixing ratios. 

### Workflow:
1. **Load and Preprocess FASTA**: Convert reference FASTA files of both host and target into integer sequences compatible with GANBase.
2. **Simulate Dataset Mixing**: Generate a synthetic testing dataset according to user-specified mixing ratios (e.g., 5:1, 10:1).
3. **Model Inference**: Pass the sequences through the GANBase discriminator to obtain prediction probabilities.
4. **Threshold Optimization**: Perform a grid search across thresholds (0.0 to 1.0) to compute performance metrics (ACC, F1, MCC, Enrichment Ratio) and output the optimal threshold.

In [ ]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, precision_score, recall_score
import torch

In [ ]:
class TextTransform:
    def __init__(self, baseseq='ACGT'):
        self.int_map = {}
        self.base_map = {}
        for ind, base in enumerate(baseseq):
            self.int_map[base] = ind
            self.base_map[ind] = base

    def text_to_int(self, text):
        """Convert DNA character text to an integer sequence"""
        int_sequence = []
        for c in text:
            if c in self.int_map:
                int_sequence.append(self.int_map[c])
            else:
                return None  # Handle non-ACGT characters or N
        return int_sequence

def load_and_process_fasta(file_path, label_name, Convert_Len=200, Min_Len=251):
    """
    Reads a FASTA file, filters sequences, trims them from index 50 to 250, 
    and converts them to integer sequences.
    """
    convert = TextTransform()
    seq_lst = []
    
    with open(file_path, 'r') as file:
        current_seq = []
        for line in file:
            if line.startswith('>'):
                if current_seq:
                    seq_str = "".join(current_seq).upper()
                    if len(seq_str) >= Min_Len:
                        # Trim sequence from 50 to 250 (length 200) as per original protocol
                        trimmed = seq_str[50:50+Convert_Len]
                        if 'N' not in trimmed:
                            int_seq = convert.text_to_int(trimmed)
                            if int_seq is not None:
                                seq_lst.append(int_seq)
                    current_seq = []
            else:
                current_seq.append(line.strip())
        # Handle the last sequence
        if current_seq:
            seq_str = "".join(current_seq).upper()
            if len(seq_str) >= Min_Len:
                trimmed = seq_str[50:50+Convert_Len]
                if 'N' not in trimmed:
                    int_seq = convert.text_to_int(trimmed)
                    if int_seq is not None:
                        seq_lst.append(int_seq)
                        
    print(f"Loaded {len(seq_lst)} valid sequences from {label_name} ({file_path}).")
    return np.array(seq_lst)

In [ ]:
def prepare_mixed_dataset(host_seqs, target_seqs, host_ratio=5, target_ratio=1, max_samples=10000):
    """
    Simulates a mixed sequencing dataset based on a predefined host:target ratio.
    Host label = 1, Target label = 0 (aligned with GANBase logic: 1 for rejection/human, 0 for keep).
    """
    # Calculate the number of samples to draw
    total_parts = host_ratio + target_ratio
    n_host = int(max_samples * (host_ratio / total_parts))
    n_target = int(max_samples * (target_ratio / total_parts))
    
    # Ensure we don't exceed available reads
    n_host = min(n_host, len(host_seqs))
    n_target = min(n_target, len(target_seqs))
    
    # Rescale to maintain exact ratio if one pool is too small
    actual_ratio_host = n_host / host_ratio
    actual_ratio_target = n_target / target_ratio
    limiting_scale = min(actual_ratio_host, actual_ratio_target)
    
    n_host = int(limiting_scale * host_ratio)
    n_target = int(limiting_scale * target_ratio)
    
    # Random sampling
    sampled_host = host_seqs[random.sample(range(len(host_seqs)), n_host)]
    sampled_target = target_seqs[random.sample(range(len(target_seqs)), n_target)]
    
    # Define labels: Host = 1, Target = 0
    host_labels = np.ones(len(sampled_host), dtype=int)
    target_labels = np.zeros(len(sampled_target), dtype=int)
    
    mixed_seqs = np.concatenate([sampled_host, sampled_target], axis=0)
    mixed_labels = np.concatenate([host_labels, target_labels], axis=0)
    
    # Shuffle the mixed dataset
    shuffle_idx = np.random.permutation(len(mixed_labels))
    return mixed_seqs[shuffle_idx], mixed_labels[shuffle_idx]

In [ ]:
def run_ganbase_inference(sequences, model_path=None):
    """
    Placeholder function for GANBase discriminator inference.
    Replace this with your actual model loading and forward pass code.
    Returns: Simulated or actual prediction probabilities (0.0 to 1.0) for Class 1 (Host).
    """
    print("Running GANBase model inference on the mixed dataset...")
    
    # --- ACUTAL MODEL CODE TEMPLATE (Uncomment and modify when deploying) ---
    # device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    # model = YourGANBaseDiscriminator().to(device)
    # model.load_state_dict(torch.load(model_path, map_location=device))
    # model.eval()
    #
    # preds = []
    # with torch.no_grad():
    #     # Batch inference
    #     for i in range(0, len(sequences), 64):
    #         batch = torch.tensor(sequences[i:i+64], dtype=torch.long).to(device)
    #         output = model(batch) # assuming output is probability for class 1
    #         preds.extend(output.cpu().numpy())
    # return np.array(preds)
    
    # Mock predictions for demonstration purposes (corresponds loosely to true labels)
    return None

In [ ]:
def evaluate_and_optimize(preds, true_labels, optimize_metric='F1-score'):
    """
    Evaluates classification metrics and the in silico enrichment ratio across 
    thresholds from 0.0 to 1.0, and selects the optimal threshold.
    """
    thresholds = np.linspace(0.01, 0.99, 99)
    results = []
    
    # Calculate initial target proportion before adaptive sequencing filter
    initial_target_count = np.sum(true_labels == 0)
    initial_host_count = np.sum(true_labels == 1)
    initial_target_prop = initial_target_count / (initial_target_count + initial_host_count)
    
    for t in thresholds:
        # According to Read Until block logic: 
        # pred >= t -> classified as 1 (human/non-target -> REJECT)
        # pred < t  -> classified as 0 (non-human/target -> KEEP)
        pred_labels = (preds >= t).astype(int)
        
        acc = accuracy_score(true_labels, pred_labels)
        f1 = f1_score(true_labels, pred_labels, zero_division=0)
        mcc = matthews_corrcoef(true_labels, pred_labels)
        precision = precision_score(true_labels, pred_labels, zero_division=0)
        recall = recall_score(true_labels, pred_labels, zero_division=0)
        
        # Calculate in silico enrichment ratio (based on kept reads: pred_labels == 0)
        kept_target = np.sum((true_labels == 0) & (pred_labels == 0))
        kept_host = np.sum((true_labels == 1) & (pred_labels == 0))
        
        if (kept_target + kept_host) > 0:
            final_target_prop = kept_target / (kept_target + kept_host)
            enrichment_ratio = final_target_prop / initial_target_prop
        else:
            enrichment_ratio = 0.0
            
        results.append({
            'Threshold': t, 'Accuracy': acc, 'F1-score': f1, 'MCC': mcc,
            'Precision': precision, 'Recall': recall, 'Enrichment_Ratio': enrichment_ratio
        })
        
    df_res = pd.DataFrame(results)
    
    # Find optimal threshold based on user preference (e.g., maximizing F1-score or MCC)
    best_idx = df_res[optimize_metric].idxmax()
    best_row = df_res.iloc[best_idx]
    
    print("\n" + "="*50)
    print(f"OPTIMIZATION SUMMARY (Optimized for maximizing {optimize_metric})")
    print(f"Optimal Threshold: {best_row['Threshold']:.3f}")
    print(f"Accuracy:          {best_row['Accuracy']:.4f}")
    print(f"F1-score:          {best_row['F1-score']:.4f}")
    print(f"MCC:               {best_row['MCC']:.4f}")
    print(f"Enrichment Ratio:  {best_row['Enrichment_Ratio']:.4f}x")
    print("="*50 + "\n")
    
    # Plot performance metrics vs threshold (Similar to Supplementary Figure 6)
    plt.figure(figsize=(10, 5))
    plt.plot(df_res['Threshold'], df_res['F1-score'], label='F1-score', color='darkorange')
    plt.plot(df_res['Threshold'], df_res['MCC'], label='MCC', color='purple')
    plt.plot(df_res['Threshold'], df_res['Accuracy'], label='Accuracy', color='green')
    plt.axvline(x=best_row['Threshold'], color='red', linestyle='--', label=f'Optimum ({best_row["Threshold"]:.2f})')
    plt.xlabel('Classification Threshold')
    plt.ylabel('Metric Value')
    plt.title('Model Performance Metrics across Different Thresholds')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.show()
    
    return best_row['Threshold'], df_res

In [ ]:
# ==========================================
# CONFIGURATION
# ==========================================
HOST_FASTA = "path/to/your/host_reference.fasta"      # Replace with your host file
TARGET_FASTA = "path/to/your/target_reference.fasta"  # Replace with your target file
HOST_RATIO = 5                                        # Parts of host data
TARGET_RATIO = 1                                      # Parts of target data
OPTIMIZATION_GOAL = 'F1-score'                        # Metric to maximize ('F1-score' or 'MCC')

# ==========================================
# PIPELINE EXECUTION
# ==========================================
if __name__ == "__main__":
    # 1. Load and process sequences
    if os.path.exists(HOST_FASTA) and os.path.exists(TARGET_FASTA):
        host_seqs = load_and_process_fasta(HOST_FASTA, "Host (Human)")
        target_seqs = load_and_process_fasta(TARGET_FASTA, "Target (Microbe)")
        
        # 2. Mix datasets according to ratio
        mixed_seqs, mixed_labels = prepare_mixed_dataset(
            host_seqs, target_seqs, host_ratio=HOST_RATIO, target_ratio=TARGET_RATIO
        )
        
        # 3. Model Inference 
        preds = run_ganbase_inference(mixed_seqs, model_path="ganbase_discriminator.pt")
    else:
        print("Reference files not found. Running demo simulation with random vectors...")
        # Demo/Mock Fallback Mode for Github repository testing
        mixed_labels = np.concatenate([np.ones(5000), np.zeros(1000)]) # 5:1 ratio mock data
        np.random.shuffle(mixed_labels)
        # Create mock predictions: host (1) usually gets high scores, target (0) gets low scores
        preds = np.where(mixed_labels == 1, np.random.beta(5, 2, 6000), np.random.beta(2, 5, 6000))
 
    # 4. Run Grid-search Threshold Optimization & Performance Plotting
    optimal_t, metrics_dataframe = evaluate_and_optimize(preds, mixed_labels, optimize_metric=OPTIMIZATION_GOAL)